# Standardization

In [1]:
import pandas as pd
import re

## Leitura do arquivo

In [3]:
df = pd.read_csv(
    "/home/tatiane/repositorios/brazil-health-data-project/data_pipeline/data/raw/ibge_populacao/tabela6579.csv",
    sep=";",
    skiprows=3
)

In [4]:
print(len(df))
df.head()

5774


,Cód.,Unidade da Federação e Município,2018,2019,2020,2021
0,11,Rondônia,1757589,1777225,1796460,1815278
1,12,Acre,869265,881935,894470,906876
2,13,Amazonas,4080611,4144597,4207714,4269995
3,14,Roraima,576568,605761,631181,652713
4,15,Pará,8513497,8602865,8690745,8777124


In [5]:
df["uf"] = df["Unidade da Federação e Município"].str.extract(r"\((\w{2})\)")
df["municipio"] = df["Unidade da Federação e Município"].str.replace(r"\s*\(\w{2}\)", "", regex=True)
df["cod_uf"] = df["Cód."].astype(str).str[:2]

- First validation

In [6]:
df.head(100)

,Cód.,Unidade da Federação e Município,2018,2019,2020,2021,uf,municipio,cod_uf
0,11,Rondônia,1757589,1777225,1796460,1815278,NaN,Rondônia,11
1,12,Acre,869265,881935,894470,906876,NaN,Acre,12
2,13,Amazonas,4080611,4144597,4207714,4269995,NaN,Amazonas,13
3,14,Roraima,576568,605761,631181,652713,NaN,Roraima,14
4,15,Pará,8513497,8602865,8690745,8777124,NaN,Pará,15
...,...,...,...,...,...,...,...,...,...
95,1200435,Santa Rosa do Purus (AC),6362,6540,6717,6893,AC,Santa Rosa do Purus,12
96,1200450,Senador Guiomard (AC),22810,23024,23236,23446,AC,Senador Guiomard,12
97,1200500,Sena Madureira (AC),45177,45848,46511,47168,AC,Sena Madureira,12
98,1200609,Tarauacá (AC),41976,42567,43151,43730,AC,Tarauacá,12


In [7]:
cols_anos = [col for col in df.columns if col.isdigit()]

df_long = df.melt(
    id_vars=["Cód.", "municipio", "uf", "cod_uf"],
    value_vars=cols_anos,
    var_name="ano",
    value_name="populacao"
)

- Second validation

In [8]:
print(len(df_long))
df_long.head(100)

23096


,Cód.,municipio,uf,cod_uf,ano,populacao
0,11,Rondônia,NaN,11,2018,1757589
1,12,Acre,NaN,12,2018,869265
2,13,Amazonas,NaN,13,2018,4080611
3,14,Roraima,NaN,14,2018,576568
4,15,Pará,NaN,15,2018,8513497
...,...,...,...,...,...,...
95,1200435,Santa Rosa do Purus,AC,12,2018,6362
96,1200450,Senador Guiomard,AC,12,2018,22810
97,1200500,Sena Madureira,AC,12,2018,45177
98,1200609,Tarauacá,AC,12,2018,41976


In [10]:
# garantir que está como string (evita problemas)
df_long['Cód.'] = df_long['Cód.'].astype(str)

# criar nova coluna de UF
df_long['cod_uf_novo'] = df_long['Cód.'].apply(lambda x: x if len(x) <= 2 else x[:2])

# criar coluna de código do município
df_long['cod_municipio'] = df_long['Cód.'].apply(lambda x: x if len(x) > 2 else None)

- Third validation

In [11]:
print(len(df_long))
df_long.head(100)

23096


,Cód.,municipio,uf,cod_uf,ano,populacao,cod_uf_novo,cod_municipio
0,11,Rondônia,NaN,11,2018,1757589,11,None
1,12,Acre,NaN,12,2018,869265,12,None
2,13,Amazonas,NaN,13,2018,4080611,13,None
3,14,Roraima,NaN,14,2018,576568,14,None
4,15,Pará,NaN,15,2018,8513497,15,None
...,...,...,...,...,...,...,...,...
95,1200435,Santa Rosa do Purus,AC,12,2018,6362,12,1200435
96,1200450,Senador Guiomard,AC,12,2018,22810,12,1200450
97,1200500,Sena Madureira,AC,12,2018,45177,12,1200500
98,1200609,Tarauacá,AC,12,2018,41976,12,1200609


In [12]:
df_long = df_long.drop(columns=['Cód.', 'cod_uf_novo'])

- Final validation

In [13]:
print(len(df_long))
df_long.head(100)

23096


,municipio,uf,cod_uf,ano,populacao,cod_municipio
0,Rondônia,NaN,11,2018,1757589,None
1,Acre,NaN,12,2018,869265,None
2,Amazonas,NaN,13,2018,4080611,None
3,Roraima,NaN,14,2018,576568,None
4,Pará,NaN,15,2018,8513497,None
...,...,...,...,...,...,...
95,Santa Rosa do Purus,AC,12,2018,6362,1200435
96,Senador Guiomard,AC,12,2018,22810,1200450
97,Sena Madureira,AC,12,2018,45177,1200500
98,Tarauacá,AC,12,2018,41976,1200609


### Writing the dataset

In [15]:
df_long.to_parquet('/home/tatiane/repositorios/brazil-health-data-project/data_pipeline/data/processed/ibge_populacao/ibge_populacao_municipios.parquet', index=False)